# Monitor Dashboard — SPX tail hedge

Read-mostly view of the current book: load a portfolio and check its health, roll status, and triggers. No position construction here — use Hedge Design for that.

In [ ]:
# Imports
import matplotlib.pyplot as plt
from IPython.display import display

from deltadewa.analysis import (
    HedgeTriggerThresholds,
    PortfolioAnalyzer,
    ScenarioGridCache,
    evaluate_hedge_triggers,
    get_volatility_stats,
)
from deltadewa.dashboard import (
    CarryDisplay,
    ChangeLogDisplay,
    PositionAgingDisplay,
    PositionDetailDisplay,
    RollStatusDisplay,
    StressDashboard,
    start_session,
)
from deltadewa.marketdata import MarketDataError
from deltadewa.reporting import PortfolioChangeTracker
from deltadewa.visualization import plot_greeks_consolidated
from deltadewa.widgets import (
    HedgeHealthDashboard,
    NetHedgeSummary,
    PortfolioWidgets,
)

In [ ]:
# Dashboard session: portfolio, market data provider, IPS policy,
# and GlobalAssumptions — all bootstrapped in one call.
# Offline-safe by default; pass use_live_market_data=True for live
# CBOE/FRED data. Export dir is set once here, from
# ctx.export_dir (default ./exports).
#
# auto_load_default=False: Monitor starts empty. Load a portfolio
# explicitly via the import widget below.
ctx = start_session(
    role="monitor",
    globals_dict=globals(),
    auto_load_default=False,
)

portfolio = ctx.portfolio
ips_config = ctx.ips_config
dashboard_config = ctx.dashboard_config
market_data = ctx.market_data
global_assumptions = ctx.global_assumptions
reporter = ctx.reporter
portfolio_changelog = ctx.changelog
portfolio_serializer = ctx.serializer
EXPORT_DIR = ctx.export_dir
today = ctx.today

portfolio_widgets = PortfolioWidgets(
    portfolio,
    portfolio_serializer,
    portfolio_changelog,
)

reporter.success("Setup complete.")

## Load portfolio

In [ ]:
# Import Widget and Portfolio Change Tracker

# Create tracker — seeds the baseline snapshot from the current portfolio state
position_tracker = PortfolioChangeTracker(
    portfolio=portfolio,
    logger=portfolio_changelog,
    reporter=reporter,
)

# Import widget — reset the tracker after a successful import so it
# doesn't diff against the pre-import snapshot
import_widget = portfolio_widgets.display_import(
    on_import_success=position_tracker.reset,
)
display(import_widget)

In [ ]:
# Volatility stats. Note: setup_dashboard()/start_session()
# already synced portfolio.volatility to the average internally;
# this just reads the stats back out.
vol_stats = get_volatility_stats(portfolio)

## Market context

In [ ]:
display(global_assumptions.display())

try:
    current_spot = market_data.get_spot(portfolio.get_symbol())
except MarketDataError:
    current_spot = global_assumptions.spot_price.value

print(f"Spot: {current_spot:,.2f}")
print(f"VIX:  {market_data.get_vix():.2f}")

## Net Hedge Summary

In [ ]:
# Hedge Summary
net_hedge_summary = NetHedgeSummary(portfolio)
display(net_hedge_summary.display())

## Hedge Health

In [ ]:
# Display Hedge Health Dashboard
health_dashboard = HedgeHealthDashboard(portfolio, config=dashboard_config)
dashboard_loader = health_dashboard.display_config_loader()
display(dashboard_loader)
display(health_dashboard.display())
# Update when portfolio changes
health_dashboard.update()

## Roll Status

In [ ]:
# Display Roll Status
if ips_config is not None:
    try:
        current_spot = market_data.get_spot(portfolio.get_symbol())
    except MarketDataError:
        current_spot = global_assumptions.spot_price.value
    RollStatusDisplay(portfolio, ips_config, reporter=reporter).display(
        current_spot,
    )

## Hedge Decision Triggers

In [ ]:
# Hedge Descision Triggers
trigger_result = evaluate_hedge_triggers(
    portfolio,
    reporter,
    thresholds=(
        HedgeTriggerThresholds.from_ips(ips_config.triggers)
        if ips_config is not None
        else None
    ),
)

## Consolidated Greeks

In [ ]:
# Consolidated Greeks Visualization
if len(portfolio.positions) > 0:
    fig = plot_greeks_consolidated(
        portfolio,
        top_n=5,
        figsize=(16, 20),
    )
    fig.subplots_adjust(hspace=0.75, wspace=0.2)
    plt.show()
else:
    print("No positions to analyze. Add positions in BUILD mode.")

## Cost of Carry

In [ ]:
# Theta Decay & Carry Analysis
carry_display = CarryDisplay(portfolio, reporter)
carry_display.display()

## Position Aging

In [ ]:
# Position Aging & Expiration Calendar
PositionAgingDisplay(portfolio, reporter).display()

## Position Detail

In [ ]:
# Position Detail Table
PositionDetailDisplay(portfolio).display()

## Current-book stress snapshot

In [ ]:
# Minimal stress setup for a single current-structure snapshot
scenario_cache = ScenarioGridCache(max_size=128)
analyzer = PortfolioAnalyzer(portfolio)
stress_dashboard = StressDashboard(
    portfolio=portfolio,
    analyzer=analyzer,
    cache=scenario_cache,
    global_assumptions=global_assumptions,
    reporter=reporter,
)

In [ ]:
# Interactive Stress Test Heatmap (Spot x Volatility)
if len(portfolio.positions) > 0:
    spot_vol_widget = stress_dashboard.create_spot_vol_heatmap(
        metric="pnl",
        days_forward=0,
    )
    display(spot_vol_widget)
else:
    reporter.error(
        "No positions to analyze. Add positions in BUILD mode first.",
    )

## Session Change Log

In [ ]:
# Portfolio Change Log Display
ChangeLogDisplay(portfolio_changelog, reporter).display()

## Export snapshot

In [ ]:
# Final Export Widget
final_export_widget = portfolio_widgets.display_export()
display(final_export_widget)

reporter.header("SESSION COMPLETE")
print("Remember to export your portfolio to save your work!")
print("Use the widget above to export in JSON, CSV, or YAML format.")
reporter.divider()